In [1]:
# -*- coding: utf-8 -*-
"""
════════════════════════════════════════════════════════════════════════════
П.4 «Итогов второй встречи с оппонентом»
Динамика частотных составляющих рядов RR и PR: общее и различия
(диапазоны 0,04–0,15 Гц и 0,16–0,4 Гц)

ЕДИНЫЙ ФАЙЛ. Ничего импортировать из соседних файлов не нужно.

Установка:   pip install numpy scipy pyedflib matplotlib
Запуск:      python П4_RR_PR_единый_код.py /путь/к/записям [--out results_p4]
Jupyter / Google Colab: выполнить ячейку целиком, затем  res = run_p4('/путь/к/записям')

Протокол пробы: около 5 мин лёжа, подъём стола до 45° без прерывания записи,
около 5 мин в положении подъёма; метки «Этап»: начало подъёма (~05:00), конец
подъёма (~05:10–05:20), конец эксперимента (~10:00). Покой — от 30 с до
(начало подъёма − 5 с); наклон — от (конец подъёма + 15 с) до (+275 с).
Отведения: каналы 0, 1, 2 = I, II, III.

Что считается
  1. Отношение НЧ/ВЧ у RR, PP и PR в покое и при наклоне; прямое сравнение
     сдвигов RR и PR, PP и PR.
  2. Абсолютные мощности в диапазонах и уровень шума измерения PR; доля шума
     в мощности PR; отношение НЧ/ВЧ у PR за вычетом шума.
  3. Погрешность измерения PR — двумя независимыми способами: по разнице PR
     одного цикла в отведениях II и III и по автокорреляции разностей PR; по
     всей записи и отдельно в покое и при наклоне (в покое зубец P ниже и
     погрешность больше, поэтому при сравнении фаз из каждой вычитается её
     собственный уровень шума); размах среднего зубца P по фазам.
  4. Когерентность RR и PR (общие колебания) с нулевым уровнем; её изменение
     при наклоне — без поправки и с поправкой на шум измерения PR;
     устойчивость к окну, отведению и уточнению измерения.
  5. Пиковые частоты в скользящем окне; проверка на синтетике, какую часть
     их «гуляния» даёт шум измерения (с погрешностью своей фазы).
  6. Проверка критерия Льюнга–Бокса на белом шуме.
  7. Опробованные варианты подготовки рядов.
════════════════════════════════════════════════════════════════════════════
"""
!pip install numpy scipy pyedflib matplotlib
import argparse, csv, glob, json, math, os, warnings
import numpy as np
import pyedflib
from scipy import stats
from scipy.signal import butter, filtfilt, find_peaks, welch, coherence
from scipy.interpolate import CubicSpline
from scipy.ndimage import median_filter
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# ═════════════════════════════ НАСТРОЙКИ ═════════════════════════════
LEAD_II_CHANNEL = 1
LEAD_III_CHANNEL = 2
LEAD_NAMES = {0: 'I', 1: 'II', 2: 'III'}
P7_LEADS = {'0011': 1, '0012': 1, '0013': 1, '0014': 1, '0015': 1, '0016': 0, '0017': 2,
            '0018': 1, '0019': 1, '0020': 2, '0021': 1, '0022': 1, '0023': 1, '0024': 1,
            '0025': 2, '0026': 1, '0027': 1, '0028': 0, '0029': 1, '0030': 1}
FS_U = 4.0                     # частота передискретизации рядов, Гц
LF = (0.04, 0.15)              # медленные колебания
HF = (0.16, 0.40)              # быстрые колебания
WELCH_SEG = 256                # 64 с — для мощностей
COH_SEG = 128                  # 32 с — для когерентности
ARTIFACT_MS = 250              # артефакт RR: отклонение от медианы 11 соседних больше 250 мс
OUTLIER_MAD = 4.0              # выброс PR/PP: быстрая часть дальше 4 MAD от медианы
COVERAGE = 0.8                 # фаза учитывается, если ряд определён хотя бы в 80 % её циклов
SLIDE_WIN_S, SLIDE_STEP_S = 64.0, 16.0


# ═════════════ ЧАСТЬ 1. Чтение, детекторы, границы фаз, уточнение ═════════════
#   (общие для всех пунктов; см. «Материалы и методы»)

def read_ecg(path, ch=0):
    f = pyedflib.EdfReader(path)
    fs = float(f.getSampleFrequency(ch))
    sig = f.readSignal(ch).astype(float)
    f.close()
    return sig, fs


def detect_r(sig, fs, band=(5.0, 20.0), integ_ms=120,
             min_rr_ms=300, win_s=10.0, k=0.35, refine_ms=100):
    x = np.asarray(sig, float)
    x = x - np.median(x)

    # 1) полосовой фильтр — полоса энергии QRS
    ny = fs/2
    b, a = butter(3, [band[0]/ny, min(band[1]/ny, 0.99)], btype='band')
    xf = filtfilt(b, a, x)

    # 2) производная + возведение в квадрат (подчёркивает крутые фронты QRS)
    d = np.diff(xf, append=xf[-1])
    sq = d**2

    # 3) интегрирование скользящим окном ~ длительность QRS
    w = max(3, int(round(integ_ms/1000*fs)))
    integ = np.convolve(sq, np.ones(w)/w, mode='same')

    # 4) ЛОКАЛЬНО адаптивный порог: доля от локального максимума
    #    (окно win_s секунд, шаг — половина окна)
    n = len(integ)
    W = max(int(win_s*fs), w*4)
    thr = np.zeros(n)
    for start in range(0, n, W//2):
        end = min(start+W, n)
        seg = integ[start:end]
        if len(seg) == 0:
            continue
        local = k*np.percentile(seg, 98)      # 98-й процентиль ≈ уровень QRS
        thr[start:end] = np.maximum(thr[start:end], local)
    thr[thr == 0] = k*np.percentile(integ, 98)

    # 5) поиск пиков с рефрактерным периодом
    dist = max(1, int(round(min_rr_ms/1000*fs)))
    cand, _ = find_peaks(integ, height=thr, distance=dist)

    # 6) уточнение положения по максимуму |полосового сигнала|
    half = max(2, int(round(refine_ms/1000*fs/2)))
    out = []
    for c in cand:
        a0, b0 = max(0, c-half), min(len(xf), c+half+1)
        out.append(a0 + int(np.argmax(np.abs(xf[a0:b0]))))
    return np.array(sorted(set(out)), dtype=int), xf


def suppress_qrs(sig, r_idx, fs, before_ms=50, after_ms=100):
    out = sig.copy()
    b = int(round(before_ms/1000*fs)); a = int(round(after_ms/1000*fs))
    for r in r_idx:
        i0, i1 = max(0, r-b), min(len(sig)-1, r+a)
        if i1-i0 < 2: continue
        out[i0:i1+1] = np.interp(np.arange(i0, i1+1), [i0, i1], [sig[i0], sig[i1]])
    return out


def halfwave(lam):
    """Вейвлет «полуволна»: положительный полупериод синуса
       с симметричными отрицательными крыльями (нулевое среднее)."""
    if lam % 2 == 0: lam += 1
    pos = np.sin(np.linspace(0, math.pi, lam))
    area = float(np.trapezoid(pos)) if hasattr(np, 'trapezoid') else float(np.trapz(pos))
    wing = max(1, lam//4)
    w = np.zeros(lam + 2*wing)
    w[:wing] = -area/(2*wing)
    w[wing:wing+lam] = pos
    w[wing+lam:] = -area/(2*wing)
    return w - w.mean()


def detect_pt(sig, r_idx, fs, wave_ms=85, thr_frac=0.3, local_s=30.0, noise_k=10.0,
              tol_ms=20.0, win=21, extra=True):
    """Возвращает индексы P и T (по одному на кардиоцикл, если найдены).
    Отклик — свёртка ЭКГ (с подавленными комплексами QRS) с вейвлетом «полуволна».
    T и основной поиск P: порог = thr_frac · 99-й процентиль положительного
    отклика по всей записи (без участков с аномально длинными RR).
    Дополнительный поиск P (для записей, где P в одной из фаз заметно меньше):
    порог по откликам в окнах поиска P в скользящем окне ±local_s секунд, но не
    ниже noise_k · уровня шума; такая находка принимается, только если расстояние
    P–R отличается от скользящей медианы по win циклам не больше tol_ms."""
    r_idx = np.asarray(r_idx, int)
    ecg = suppress_qrs(sig, r_idx, fs)
    lam = max(5, int(round(wave_ms/1000*fs)))
    w = halfwave(lam)
    conv = np.convolve(ecg, w[::-1], mode='same')
    n = len(conv)

    # порог по «здоровым» участкам (исключаем аномально длинные RR)
    rr = np.diff(r_idx)
    med = np.median(rr) if len(rr) else 0
    ok = np.ones(n, bool)
    for k in range(len(rr)):
        if med and rr[k] > 1.8*med:
            ok[r_idx[k]:r_idx[k+1]] = False
    pos = conv[(conv > 0) & ok]
    thr = thr_frac*np.percentile(pos, 99) if len(pos) else 0

    # окна поиска P и T
    wins = []
    for k in range(len(r_idx)):
        r = r_idx[k]
        r_prev = r_idx[k-1] if k > 0 else max(0, r-int(fs))
        r_next = r_idx[k+1] if k < len(r_idx)-1 else min(len(sig)-1, r+int(fs))
        # P: непосредственно перед QRS (P предшествует R на 100–300 мс)
        rr_p = r - r_prev
        lo = max(r_prev + rr_p//2, r - int(0.35*fs))
        ps, pe = lo, r - max(1, int(0.06*fs))
        # T: после QRS, до начала следующего цикла
        rr_n = r_next - r
        ts = r + max(int(0.12*fs), rr_n//8)
        te = min(r + int(0.6*rr_n), r_next - int(0.08*fs))
        wins.append((ps, pe, ts, te))

    P, T = [], []
    for ps, pe, ts, te in wins:
        if ps < pe <= len(conv):
            seg = conv[ps:pe]
            P.append(ps + int(np.argmax(seg)) if len(seg) and seg.max() > thr else -1)
        else:
            P.append(-1)
        if ts < te <= len(conv):
            seg = conv[ts:te]
            T.append(ts + int(np.argmax(seg)) if len(seg) and seg.max() > thr else -1)
        else:
            T.append(-1)
    P = np.array(P); T = np.array(T)

    if not extra:
        return P, T, ecg, conv
    # дополнительный поиск P: порог по окнам P в скользящем окне со страховкой от шума
    pmask = np.zeros(n, bool)
    for ps, pe, _, _ in wins:
        if ps < pe <= n:
            pmask[ps:pe] = True
    d = np.diff(ecg)
    noise = 1.4826*np.median(np.abs(d - np.median(d)))/np.sqrt(2)*np.sqrt(np.sum(w**2))
    L = int(local_s*fs)
    P2 = np.full(len(r_idx), -1)
    for k, (ps, pe, _, _) in enumerate(wins):
        if not (ps < pe <= n):
            continue
        i = r_idx[k]; a, b = max(0, i-L), min(n, i+L)
        sg = conv[a:b]; mk = (sg > 0) & ok[a:b] & pmask[a:b]
        thr_p = thr_frac*np.percentile(sg[mk], 99) if mk.sum() > 10 else thr
        seg = conv[ps:pe]
        if len(seg) and seg.max() > max(thr_p, noise_k*noise):
            P2[k] = ps + int(np.argmax(seg))
    off = np.where(P2 >= 0, (r_idx - P2)/fs*1000, np.nan)
    med_off = np.full(len(off), np.nan); h = win//2
    for i in range(len(off)):
        s = off[max(0, i-h):i+h+1]; s = s[np.isfinite(s)]
        if len(s) >= 3:
            med_off[i] = np.median(s)
    add = (P < 0) & (P2 >= 0) & (np.abs(off - med_off) <= tol_ms)
    P[add] = P2[add]
    return P, T, ecg, conv


def phase_bounds(marks):
    """Границы подъёма стола по протоколу пробы. Метки «Этап»: около 05:00 —
    начало подъёма, около 05:10–05:20 — конец подъёма, около 10:00 — конец
    записи. Если в зоне подъёма несколько меток, первая — начало, последняя —
    конец; если одна и она позже 05:10 — это конец подъёма, начало принимается
    на 15 с раньше. Возвращает (начало подъёма, конец подъёма), с."""
    mid = sorted(t for t in marks if 250 < t < 340)
    if len(mid) >= 2:
        return mid[0], mid[-1]
    if len(mid) == 1:
        m = mid[0]
        return (m - 15.0, m) if m >= 310 else (m, m + 15.0)
    return 300.0, 315.0


def band_ecg(sig,fs,lo=0.5,hi=40.0):
    b,a=butter(2,[lo/(fs/2),hi/(fs/2)],'band'); return filtfilt(b,a,sig)


def parabolic(y,i):
    """Вершина параболы через три точки вокруг i: поправка в долях отсчёта."""
    if i<=0 or i>=len(y)-1: return float(i)
    a,b,c=y[i-1],y[i],y[i+1]; den=a-2*b+c
    return float(i) if den==0 else i+0.5*(a-c)/den


def refine_r(y,r,fs,half_ms=8):
    """Уточнение вершины R: максимум |ЭКГ| рядом с отметкой + парабола."""
    h=int(half_ms/1000*fs); out=np.full(len(r),np.nan)
    for k,ri in enumerate(r):
        a,b=max(1,ri-h),min(len(y)-1,ri+h+1); seg=np.abs(y[a:b]); i=int(np.argmax(seg))
        out[k]=a+parabolic(seg,i)
    return out


def refine_p(y,P,fs,half_ms=60,search_ms=20,iters=2):
    """Уточнение положения P по шаблону: шаблон — среднее участков ЭКГ вокруг
    найденных P; в каждом цикле — сдвиг максимальной корреляции с шаблоном
    (в пределах ±search_ms) с параболической интерполяцией."""
    P=np.asarray(P); h=int(half_ms/1000*fs); s=int(search_ms/1000*fs); idx=np.where(P>=0)[0]
    pos=P.astype(float).copy(); pos[P<0]=np.nan
    for _ in range(iters):
        segs=[]
        for k in idx:
            c=int(round(pos[k]))
            if c-h-s>=0 and c+h+s<len(y): segs.append(y[c-h:c+h+1]-np.mean(y[c-h:c+h+1]))
        if len(segs)<20: return pos
        tpl=np.mean(segs,0); tpl=tpl-tpl.mean(); tn=np.linalg.norm(tpl)
        new=pos.copy()
        for k in idx:
            c=int(round(pos[k]))
            if not (c-h-s>=0 and c+h+s<len(y)): continue
            cc=np.empty(2*s+1)
            for j,l in enumerate(range(-s,s+1)):
                w=y[c+l-h:c+l+h+1]; w=w-w.mean(); nw=np.linalg.norm(w)
                cc[j]=np.dot(w,tpl)/(nw*tn) if nw>0 else -1
            i=int(np.argmax(cc)); new[k]=c+(parabolic(cc,i)-s)
        pos=new
    return pos


def build_series_ch(path, ch, refine=True, extra_p=True):
    """Ряды RR, PP и PR одного отведения. Положения R и P уточняются точнее
    одного отсчёта: R — параболической интерполяцией вершины, P — по шаблону
    зубца P записи (корреляция в пределах ±20 мс с интерполяцией). Все три ряда
    строятся из одних и тех же уточнённых точек, поэтому тождество
    RR(k) = PP(k) + PR(k+1) − PR(k) выполняется точно."""
    f = pyedflib.EdfReader(path)
    ann = f.readAnnotations()
    f.close()
    marks = [float(t) for t in ann[0]]
    p1e, p2b = phase_bounds(marks)

    sig, fs = read_ecg(path, ch=ch)
    r, _ = detect_r(sig, fs)
    P, T, _, _ = detect_pt(sig, r, fs, extra=extra_p)
    if refine:
        y = band_ecg(sig, fs)
        r_ref = refine_r(y, r, fs)
        p_ref = refine_p(y, P, fs)
    else:                                   # вариант без уточнения: точки — индексы отсчётов
        r_ref = r.astype(float)
        p_ref = np.where(P >= 0, P, np.nan).astype(float)

    t_r = r_ref/fs
    rr = np.diff(r_ref)/fs*1000
    has_p = (P > 0) & np.isfinite(p_ref)
    pr = np.full(len(r), np.nan)
    pr[has_p] = (r_ref[has_p] - p_ref[has_p])/fs*1000
    pp = np.full(len(r), np.nan)
    idx = np.where(has_p)[0]
    for a, b in zip(idx[:-1], idx[1:]):
        if b == a + 1:
            pp[a] = (p_ref[b] - p_ref[a])/fs*1000

    return dict(t_r=t_r, rr=rr, t_rr=t_r[:-1], pr=pr, pp=pp, r_ref=r_ref, p_ref=p_ref, fs=fs,
                p1e=p1e, p2b=p2b, n_r=len(r),
                n_pr=int(np.isfinite(pr).sum()), n_pp=int(np.isfinite(pp).sum()))


# ═════════════ ЧАСТЬ 2. Подготовка рядов ═════════════

def phase_windows(v):
    return {'покой': (30.0, v['p1e'] - 5.0), 'наклон': (v['p2b'] + 15.0, v['p2b'] + 275.0)}


def clean_rr(rr):
    x = np.asarray(rr, float).copy(); base = np.where(np.isfinite(x), x, np.nanmedian(x))
    x[np.abs(x - median_filter(base, 11, mode='nearest')) > ARTIFACT_MS] = np.nan
    return x


def clean_outliers(x):
    x = np.asarray(x, float).copy(); ok = np.isfinite(x)
    if ok.sum() < 30:
        return x
    xs = x[ok]; f = xs - np.convolve(np.pad(xs, 7, mode='edge'), np.ones(15) / 15, 'valid')
    med = np.median(f); mad = 1.4826 * np.median(np.abs(f - med)); idx = np.where(ok)[0]
    x[idx[np.abs(f - med) > OUTLIER_MAD * mad]] = np.nan
    return x


def to_uniform(t, x, lo, hi, cov=COVERAGE):
    """Ряд на равномерной сетке FS_U внутри окна [lo, hi]: кубический сплайн по
    циклам с известным значением, затем удаление линейного тренда. None —
    если ряд определён менее чем в cov циклов окна."""
    mk = (t >= lo) & (t <= hi); fin = mk & np.isfinite(x)
    if fin.sum() < 20 or (cov and fin.sum() / max(1, mk.sum()) < cov):
        return None
    tu = np.arange(t[fin][0], t[fin][-1], 1 / FS_U); u = CubicSpline(t[fin], x[fin])(tu)
    tt = np.arange(len(u)) / FS_U
    return u - np.polyval(np.polyfit(tt, u, 1), tt)


def band_power(u, band, seg=WELCH_SEG):
    f, P = welch(u, fs=FS_U, nperseg=min(seg, len(u)))
    mk = (f >= band[0]) & (f < band[1]); return float(np.trapezoid(P[mk], f[mk]))


def prepare(series, clean=True, cov=COVERAGE, windows='protocol'):
    """Равномерные ряды RR, PP, PR по фазам для каждой записи."""
    out = {}
    for n, v in series.items():
        rr = clean_rr(v['rr']) if clean else np.asarray(v['rr'], float)
        pr = clean_outliers(v['pr']) if clean else np.asarray(v['pr'], float)
        pp = clean_outliers(v['pp']) if clean else np.asarray(v['pp'], float)
        W = phase_windows(v)
        if windows == 'whole':
            W = {'покой': (v['t_r'][0], v['p1e']), 'наклон': (v['p2b'], v['t_r'][-1])}
        out[n] = {}
        for ph, (lo, hi) in W.items():
            out[n][ph] = dict(RR=to_uniform(v['t_rr'], rr, lo, hi, 0.0), PP=to_uniform(v['t_r'], pp, lo, hi, cov),
                              PR=to_uniform(v['t_r'], pr, lo, hi, cov),
                              n_beats=int(((v['t_r'] >= lo) & (v['t_r'] <= hi)).sum()))
    return out


# ═════════════ ЧАСТЬ 3. Отношение НЧ/ВЧ и прямое сравнение сдвигов ═════════════

def lfhf_analysis(U):
    rows = {}
    for n, d in U.items():
        rows[n] = {ph: {k: (band_power(d[ph][k], LF) / band_power(d[ph][k], HF) if d[ph][k] is not None else None)
                        for k in ('RR', 'PP', 'PR')} for ph in d}
    out = {}
    for k in ('RR', 'PP', 'PR'):
        ok = [n for n in rows if rows[n]['покой'][k] is not None and rows[n]['наклон'][k] is not None]
        r = np.array([rows[n]['покой'][k] for n in ok]); t = np.array([rows[n]['наклон'][k] for n in ok])
        out[k] = dict(n=len(ok), rest=float(np.median(r)), tilt=float(np.median(t)), ratio=float(np.median(t / r)),
                      up=int((t > r).sum()), p=float(stats.wilcoxon(r, t)[1]))
    for ref in ('RR', 'PP'):
        ok = [n for n in rows if all(rows[n][ph][k] is not None for ph in ('покой', 'наклон') for k in (ref, 'PR'))]
        d1 = np.array([np.log(rows[n]['наклон'][ref] / rows[n]['покой'][ref]) for n in ok])
        d2 = np.array([np.log(rows[n]['наклон']['PR'] / rows[n]['покой']['PR']) for n in ok])
        out[f'{ref}–PR'] = dict(n=len(ok), p=float(stats.wilcoxon(d1, d2)[1]))
    return out, rows


# ═════════════ ЧАСТЬ 4. Погрешность измерения PR ═════════════

def match_beats(ra, rb, fs, tol_ms=40):
    j = np.searchsorted(rb, ra); j = np.clip(j, 1, len(rb) - 1)
    j = np.where(np.abs(rb[j - 1] - ra) < np.abs(rb[j] - ra), j - 1, j)
    return j, np.abs(rb[j] - ra) <= tol_ms / 1000 * fs


def crosslead_noise(v2, v3):
    """Погрешность PR на одно отведение: робастное СКО поцикловой разницы PR
    в отведениях II и III, делённое на √2 (оценка сверху)."""
    j, ok = match_beats(v2['r_ref'], v3['r_ref'], v2['fs'])
    d = v2['pr'][ok] - v3['pr'][j[ok]]; d = d[np.isfinite(d)]; d = d - np.median(d)
    mad = 1.4826 * np.median(np.abs(d)); d = d[np.abs(d) <= 5 * max(mad, 1.0)]
    return float(np.std(d) / np.sqrt(2))


def decomposition_noise(v):
    """Разложение дисперсии разностей соседних PR: для белого шума автокорреляция
    разностей на первом шаге равна −0,5; доля шума = −r1 / 0,5. Если r1 ≥ 0,
    медленные изменения PR перекрывают вклад шума, и оценка этим способом не
    определена (NaN); такие записи в медианы не входят."""
    pr = np.asarray(v['pr'], float); d = pr[1:] - pr[:-1]; ok = np.isfinite(d[:-1]) & np.isfinite(d[1:])
    a, b = d[:-1][ok], d[1:][ok]; q = np.percentile(np.abs(np.r_[a, b]), 99); keep = (np.abs(a) <= q) & (np.abs(b) <= q)
    a, b = a[keep], b[keep]; r1 = float(np.corrcoef(a, b)[0, 1]); sd = float(np.std(np.r_[a, b], ddof=1))
    frac = min(1.0, -r1 / 0.5) if r1 < 0 else np.nan
    return dict(r1=r1, sd_diff=sd, frac=frac, noise_sd=sd * np.sqrt(frac) / np.sqrt(2), own_sd=sd * np.sqrt(1 - frac))


def phase_noise(v2, v3):
    """Погрешность PR отдельно в покое и при наклоне — те же две оценки, что по
    всей записи, но только по циклам фазы. В покое зубец P ниже и погрешность
    больше, поэтому одна оценка на всю запись для сравнения фаз не годится.
    Если в фазе r1 ≥ 0 (медленные изменения PR маскируют шум), оценка по
    разложению для неё не определена (NaN)."""
    W = phase_windows(v2); out = {}
    for ph, (lo, hi) in W.items():
        m = (v2['t_r'] >= lo) & (v2['t_r'] <= hi)
        pr = np.asarray(v2['pr'], float); d = pr[1:] - pr[:-1]; d[~(m[1:] & m[:-1])] = np.nan
        ok = np.isfinite(d[:-1]) & np.isfinite(d[1:]); dec = np.nan
        if ok.sum() >= 50:
            a, b = d[:-1][ok], d[1:][ok]; q = np.percentile(np.abs(np.r_[a, b]), 99); keep = (np.abs(a) <= q) & (np.abs(b) <= q)
            r1 = float(np.corrcoef(a[keep], b[keep])[0, 1]); sd = float(np.std(np.r_[a[keep], b[keep]], ddof=1))
            if r1 < 0:
                dec = sd * np.sqrt(min(1.0, -r1 / 0.5)) / np.sqrt(2)
        j, okb = match_beats(v2['r_ref'], v3['r_ref'], v2['fs']); okb = okb & m
        dd = v2['pr'][okb] - v3['pr'][j[okb]]; dd = dd[np.isfinite(dd)]; dd = dd - np.median(dd)
        mad = 1.4826 * np.median(np.abs(dd)); dd = dd[np.abs(dd) <= 5 * max(mad, 1.0)]
        out[ph] = dict(decomp=float(dec), crosslead=float(np.std(dd) / np.sqrt(2)))
    return out


def p_amplitude(path, ch, v):
    """Размах среднего зубца P в каждой фазе: ЭКГ 0,5–40 Гц, участки ±60 мс вокруг
    уточнённой точки P, базовая линия — первые 15 мс участка."""
    sig, fs = read_ecg(path, ch); y = band_ecg(sig, fs); h = int(0.06 * fs); out = {}
    for ph, (lo, hi) in phase_windows(v).items():
        segs = []
        for k in np.where(np.isfinite(v['p_ref']) & (v['t_r'] >= lo) & (v['t_r'] <= hi))[0]:
            c = int(round(v['p_ref'][k]))
            if c - h >= 0 and c + h < len(y):
                sg = y[c - h:c + h + 1]; segs.append(sg - np.mean(sg[:int(0.015 * fs)]))
        out[ph] = float(np.ptp(np.mean(segs, 0))) if len(segs) >= 20 else np.nan
    return out


def noise_floor(n_beats, length, sd, band, reps=30, seed=0):
    """Мощность в диапазоне, которую дал бы один шум измерения с СКО sd,
    заданный по моментам сокращений и обработанный так же, как ряд."""
    rng = np.random.default_rng(seed); vals = []
    for _ in range(reps):
        tb = np.linspace(0, length / FS_U, max(n_beats, 20)); z = rng.standard_normal(len(tb)) * sd
        tu = np.arange(length) / FS_U; u = CubicSpline(tb, z)(tu); tt = tu
        u = u - np.polyval(np.polyfit(tt, u, 1), tt); vals.append(band_power(u, band))
    return float(np.median(vals))


def power_analysis(U, noise_sd):
    """Абсолютные мощности RR и PR по фазам; ВЧ PR за вычетом уровня шума."""
    rec = {}
    for n, d in U.items():
        if any(d[ph]['PR'] is None or not np.isfinite(noise_sd[(n, ph)]) for ph in d):
            continue
        r = {}
        for ph in d:
            u = d[ph]['PR']; fl = noise_floor(d[ph]['n_beats'], len(u), noise_sd[(n, ph)], HF)
            fl_lf = noise_floor(d[ph]['n_beats'], len(u), noise_sd[(n, ph)], LF)
            r[ph] = dict(rr_lf=band_power(d[ph]['RR'], LF), rr_hf=band_power(d[ph]['RR'], HF),
                         pr_lf=band_power(u, LF), pr_hf=band_power(u, HF), floor_hf=fl, floor_lf=fl_lf)
        rec[n] = r
    ok = list(rec); out = {}
    for k in ('rr_lf', 'rr_hf', 'pr_lf', 'pr_hf'):
        a = np.array([rec[n]['покой'][k] for n in ok]); b = np.array([rec[n]['наклон'][k] for n in ok])
        out[k] = dict(n=len(ok), rest=float(np.median(a)), tilt=float(np.median(b)), down=int((b < a).sum()), p=float(stats.wilcoxon(a, b)[1]))
    a = np.array([rec[n]['покой']['pr_hf'] - rec[n]['покой']['floor_hf'] for n in ok])
    b = np.array([rec[n]['наклон']['pr_hf'] - rec[n]['наклон']['floor_hf'] for n in ok])
    out['pr_hf_minus_noise'] = dict(n=len(ok), rest=float(np.median(a)), tilt=float(np.median(b)), down=int((b < a).sum()), p=float(stats.wilcoxon(a, b)[1]))
    out['pr_hf_over_noise'] = {ph: float(np.median([rec[n][ph]['pr_hf'] / rec[n][ph]['floor_hf'] for n in ok])) for ph in ('покой', 'наклон')}
    # доля шума в мощности PR по диапазонам
    sh = lambda n, ph, b: min(rec[n][ph][f'floor_{b}'] / rec[n][ph][f'pr_{b}'], 0.95)
    for b in ('lf', 'hf'):
        a = np.array([sh(n, 'покой', b) for n in ok]); c = np.array([sh(n, 'наклон', b) for n in ok])
        out[f'noise_share_{b}'] = dict(n=len(ok), rest=float(np.median(a)), tilt=float(np.median(c)), down=int((c < a).sum()), p=float(stats.wilcoxon(a, c)[1]))
    # НЧ/ВЧ PR за вычетом шума (не меньше 5 % исходной мощности) и сравнение его сдвига со сдвигом RR
    cr = lambda n, ph: max(rec[n][ph]['pr_lf'] - rec[n][ph]['floor_lf'], 0.05 * rec[n][ph]['pr_lf']) / max(rec[n][ph]['pr_hf'] - rec[n][ph]['floor_hf'], 0.05 * rec[n][ph]['pr_hf'])
    a = np.array([cr(n, 'покой') for n in ok]); c = np.array([cr(n, 'наклон') for n in ok])
    lr = np.log(np.array([(rec[n]['наклон']['rr_lf'] / rec[n]['наклон']['rr_hf']) / (rec[n]['покой']['rr_lf'] / rec[n]['покой']['rr_hf']) for n in ok])); lp = np.log(c / a)
    out['pr_lfhf_minus_noise'] = dict(n=len(ok), rest=float(np.median(a)), tilt=float(np.median(c)), ratio=float(np.exp(np.median(lp))), up=int((c > a).sum()), p=float(stats.wilcoxon(a, c)[1]))
    out['rr_vs_pr_minus_noise'] = dict(n=len(ok), rr_stronger=int((lr > lp).sum()), p=float(stats.wilcoxon(lr, lp)[1]))
    out['shares'] = {n: {ph: (sh(n, ph, 'lf'), sh(n, ph, 'hf')) for ph in ('покой', 'наклон')} for n in ok}
    return out


def coherence_noise_corrected(rec, shares):
    """Когерентность с поправкой на шум измерения PR: шум, не связанный с RR,
    уменьшает когерентность в (1 − доля шума) раз, поэтому наблюдаемая делится
    на (1 − доля шума в мощности PR в этом диапазоне)."""
    both = [n for n in shares if (n, 'покой') in rec and (n, 'наклон') in rec]; out = {}
    for i, bn in enumerate(('LF', 'HF')):
        c = {ph: np.array([min(1.0, rec[(n, ph)][bn] / (1 - shares[n][ph][i])) for n in both]) for ph in ('покой', 'наклон')}
        out[bn] = dict(n=len(both), rest=float(np.median(c['покой'])), tilt=float(np.median(c['наклон'])),
                       up=int((c['наклон'] > c['покой']).sum()), p=float(stats.wilcoxon(c['покой'], c['наклон'])[1]))
    return out


# ═════════════ ЧАСТЬ 5. Когерентность RR и PR ═════════════

def _bm(f, c, band):
    mk = (f >= band[0]) & (f <= band[1]); return float(np.mean(c[mk]))


def coherence_analysis(U, seg=COH_SEG, with_null=True):
    rec = {}
    for n, d in U.items():
        for ph in d:
            a, b = d[ph]['RR'], d[ph]['PR']
            if a is None or b is None:
                continue
            L = min(len(a), len(b)); f, c = coherence(a[:L], b[:L], fs=FS_U, nperseg=seg)
            _, Pa = welch(a[:L], fs=FS_U, nperseg=seg); _, Pb = welch(b[:L], fs=FS_U, nperseg=seg); mk = (f >= HF[0]) & (f <= HF[1])
            rec[(n, ph)] = dict(LF=_bm(f, c, LF), HF=_bm(f, c, HF), pk_rr=float(f[mk][np.argmax(Pa[mk])]), pk_pr=float(f[mk][np.argmax(Pb[mk])]))
    out = {'по_фазам': {}, 'изменение': {}}
    for ph in ('покой', 'наклон'):
        have = [n for (n, p) in rec if p == ph]
        for band, bn in ((LF, 'LF'), (HF, 'HF')):
            v = np.array([rec[(n, ph)][bn] for n in have]); e = dict(n=len(v), median=float(np.median(v)))
            if with_null:
                nl = []
                for n1 in have:
                    for n2 in have:
                        if n1 == n2:
                            continue
                        a = U[n1][ph]['RR']; b = U[n2][ph]['PR']; L = min(len(a), len(b))
                        f, c = coherence(a[:L], b[:L], fs=FS_U, nperseg=seg); nl.append(_bm(f, c, band))
                nl = np.array(nl); thr = float(np.percentile(nl, 95))
                e.update(null_median=float(np.median(nl)), null95=thr, above=int((v > thr).sum()))
            out['по_фазам'][f'{ph}|{bn}'] = e
        a = np.array([rec[(n, ph)]['pk_rr'] for n in have]); b = np.array([rec[(n, ph)]['pk_pr'] for n in have])
        out['по_фазам'][f'{ph}|пик_ВЧ'] = dict(n=len(a), rr=float(np.median(a)), pr=float(np.median(b)), match=int((np.abs(a - b) <= 0.0313).sum()))
    both = [n for n in U if (n, 'покой') in rec and (n, 'наклон') in rec]
    for bn in ('LF', 'HF'):
        r = np.array([rec[(n, 'покой')][bn] for n in both]); t = np.array([rec[(n, 'наклон')][bn] for n in both])
        out['изменение'][bn] = dict(n=len(both), rest=float(np.median(r)), tilt=float(np.median(t)), up=int((t > r).sum()), p=float(stats.wilcoxon(r, t)[1]))
    return out, rec


# ═════════════ ЧАСТЬ 6. Пиковые частоты во времени и шум ═════════════

def _peak_track(x, band):
    W = int(SLIDE_WIN_S * FS_U); S = int(SLIDE_STEP_S * FS_U); pk = []
    for st in range(0, len(x) - W + 1, S):
        f, P = welch(x[st:st + W], fs=FS_U, nperseg=W, nfft=4 * W); mk = (f >= band[0]) & (f <= band[1]); pk.append(f[mk][np.argmax(P[mk])])
    return np.array(pk)


def _bandpass_fft(x, band):
    X = np.fft.rfft(x); f = np.fft.rfftfreq(len(x), 1 / FS_U); X[(f < band[0]) | (f > band[1])] = 0
    return np.fft.irfft(X, len(x))


def peak_dynamics(U, noise_sd, reps=10, seed=0):
    rng = np.random.default_rng(seed); rows = []
    for n, d in U.items():
        for ph in d:
            a, b = d[ph]['RR'], d[ph]['PR']
            if a is None or b is None or not np.isfinite(noise_sd[(n, ph)]):
                continue
            L = min(len(a), len(b)); a, b = a[:L], b[:L]
            pa, pb = _peak_track(a, HF), _peak_track(b, HF)
            aH = _bandpass_fft(a, HF); Pa = band_power(aH, HF, COH_SEG); Pb = band_power(b, HF, COH_SEG); sims = []
            for _ in range(reps):
                tb = np.linspace(0, L / FS_U, max(d[ph]['n_beats'], 20)); z = rng.standard_normal(len(tb)) * noise_sd[(n, ph)]
                nz = CubicSpline(tb, z)(np.arange(L) / FS_U); nz -= nz.mean(); Pn = band_power(nz, HF, COH_SEG)
                k = np.sqrt(max(Pb - Pn, 0) / Pa) if Pa > 0 else 0.0; sims.append(np.std(_peak_track(k * aH + nz, HF)))
            rho = stats.spearmanr(pa, pb)[0] if np.std(pa) > 0 and np.std(pb) > 0 else np.nan
            rows.append(dict(record=n, phase=ph, sd_rr=float(np.std(pa)), sd_pr=float(np.std(pb)), sd_sim=float(np.median(sims)), rho=float(rho), windows=len(pa)))
    out = {}
    for ph in ('покой', 'наклон'):
        R = [r for r in rows if r['phase'] == ph]; rho = [r['rho'] for r in R if np.isfinite(r['rho'])]
        out[ph] = dict(n=len(R), sd_rr=float(np.median([r['sd_rr'] for r in R])), sd_pr=float(np.median([r['sd_pr'] for r in R])),
                       sd_sim=float(np.median([r['sd_sim'] for r in R])), pr_gt_sim=int(sum(r['sd_pr'] > r['sd_sim'] for r in R)),
                       rho=float(np.median(rho)), rho_pos=int(sum(x > 0 for x in rho)), n_rho=len(rho), windows=int(np.median([r['windows'] for r in R])))
        # знаковый критерий: чаще ли пик настоящего PR смещается сильнее, чем у искусственного ряда
        out[ph]['p_sign'] = float(stats.binomtest(out[ph]['pr_gt_sim'], out[ph]['n']).pvalue)
    return out


# ═════════════ ЧАСТЬ 7. Проверка критерия Льюнга–Бокса ═════════════

def ljung_box_p(x, h=10):
    x = np.asarray(x, float); x = x[np.isfinite(x)]; n = len(x); x = x - x.mean(); c0 = (x * x).sum()
    q = n * (n + 2) * sum(((x[:-k] * x[k:]).sum() / c0) ** 2 / (n - k) for k in range(1, h + 1))
    return float(1 - stats.chi2.cdf(q, h))


def ljung_box_check(reps=200, n=700, seed=0):
    rng = np.random.default_rng(seed); r = {'без обработки': 0, 'разности соседних значений': 0, 'вычитание скользящего среднего': 0}
    for _ in range(reps):
        w = rng.standard_normal(n)
        r['без обработки'] += ljung_box_p(w) < 0.05; r['разности соседних значений'] += ljung_box_p(np.diff(w)) < 0.05
        r['вычитание скользящего среднего'] += ljung_box_p(w[7:-7] - np.convolve(w, np.ones(15) / 15, 'valid')) < 0.05
    return {k: v / reps for k, v in r.items()}


# ═════════════ ЧАСТЬ 8. Рисунки ═════════════

def _comma(fig):
    from matplotlib.ticker import FuncFormatter
    fmt = FuncFormatter(lambda v, _: f'{v:g}'.replace('.', ','))
    for a in fig.axes:
        if a.get_yscale() == 'linear':
            a.yaxis.set_major_formatter(fmt)


def fig_lfhf(rows, path):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)
    for k, (key, title) in enumerate((('RR', 'а — RR'), ('PR', 'б — PR'))):
        for n, d in rows.items():
            if d['покой'][key] is None or d['наклон'][key] is None:
                continue
            ax[k].plot([0, 1], [d['покой'][key], d['наклон'][key]], '-o', color='#1565c0' if key == 'RR' else '#c62828', alpha=.6, ms=4)
        ax[k].set_xticks([0, 1]); ax[k].set_xticklabels(['покой', 'наклон']); ax[k].set_yscale('log'); ax[k].set_title(title); ax[k].grid(alpha=.3)
    ax[0].set_ylabel('НЧ/ВЧ'); fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def fig_coh(rec, path):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)
    names = sorted({n for n, _ in rec})
    for k, (bn, title) in enumerate((('LF', 'а — НЧ (0,04–0,15 Гц)'), ('HF', 'б — ВЧ (0,16–0,4 Гц)'))):
        for n in names:
            if (n, 'покой') in rec and (n, 'наклон') in rec:
                ax[k].plot([0, 1], [rec[(n, 'покой')][bn], rec[(n, 'наклон')][bn]], '-o', color='#2e7d32', alpha=.6, ms=4)
        ax[k].set_xticks([0, 1]); ax[k].set_xticklabels(['покой', 'наклон']); ax[k].set_title(title); ax[k].grid(alpha=.3); ax[k].set_ylim(0, 1)
    ax[0].set_ylabel('когерентность RR и PR'); _comma(fig); fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


# ═════════════ ЧАСТЬ 9. Общий запуск ═════════════

def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    return o


def run_p4(data_dir, out_dir='results_p4'):
    os.makedirs(out_dir, exist_ok=True)
    paths = sorted(glob.glob(os.path.join(data_dir, '00*_ecg.edf'))); names = [os.path.basename(p)[:4] for p in paths]
    if not paths:
        raise SystemExit(f'В папке {data_dir} нет файлов 00*_ecg.edf')
    print(f'Записей: {len(paths)}. Строю ряды (отведения II и III, варианты измерения)...')
    B = lambda ch_fn, **kw: {n: build_series_ch(p, ch_fn(n), **kw) for n, p in zip(names, paths)}
    S = {'II': B(lambda n: LEAD_II_CHANNEL), 'III': B(lambda n: LEAD_III_CHANNEL),
         'II_raw': B(lambda n: LEAD_II_CHANNEL, refine=False), 'III_raw': B(lambda n: LEAD_III_CHANNEL, refine=False),
         'II_main_thr': B(lambda n: LEAD_II_CHANNEL, extra_p=False), 'best': B(lambda n: P7_LEADS[n])}
    summary = {}
    # погрешность
    dec = {n: decomposition_noise(S['II'][n]) for n in names}; dec_raw = {n: decomposition_noise(S['II_raw'][n]) for n in names}
    cl = {n: crosslead_noise(S['II'][n], S['III'][n]) for n in names}; cl_raw = {n: crosslead_noise(S['II_raw'][n], S['III_raw'][n]) for n in names}
    med = lambda D, k=None: float(np.nanmedian([d[k] if k else d for d in D.values()]))
    summary['noise'] = dict(decomp_undefined=[n for n in names if not np.isfinite(dec[n]['noise_sd'])], decomp_raw_undefined=[n for n in names if not np.isfinite(dec_raw[n]['noise_sd'])],
                            decomp=dict(r1=float(np.mean([d['r1'] for d in dec.values()])), noise_sd=med(dec, 'noise_sd'), own_sd=med(dec, 'own_sd')),
                            decomp_raw=dict(r1=float(np.mean([d['r1'] for d in dec_raw.values()])), noise_sd=med(dec_raw, 'noise_sd'), own_sd=med(dec_raw, 'own_sd')),
                            crosslead=med(cl), crosslead_raw=med(cl_raw))
    PN = {n: phase_noise(S['II'][n], S['III'][n]) for n in names}
    nz_dec = {(n, ph): PN[n][ph]['decomp'] for n in names for ph in PN[n]}; nz_cl = {(n, ph): PN[n][ph]['crosslead'] for n in names for ph in PN[n]}
    U = prepare(S['II']); both = [n for n in names if all(U[n][ph]['PR'] is not None for ph in U[n])]
    pn = {'records': both}
    for k, lab in (('decomp', 'разложение'), ('crosslead', 'разница II–III')):
        ok = [n for n in both if np.isfinite(PN[n]['покой'][k]) and np.isfinite(PN[n]['наклон'][k])]
        a = np.array([PN[n]['покой'][k] for n in ok]); b = np.array([PN[n]['наклон'][k] for n in ok])
        pn[k] = dict(n=len(ok), rest=float(np.median(a)), tilt=float(np.median(b)), down=int((b < a).sum()), p=float(stats.wilcoxon(a, b)[1]))
        print(f"Погрешность PR по фазам ({lab}, записи с обеими фазами): покой {pn[k]['rest']:.2f} → наклон {pn[k]['tilt']:.2f} мс, "
              f"меньше при наклоне у {pn[k]['down']}/{pn[k]['n']}, p = {pn[k]['p']:.4f}")
    pn['decomp_undefined'] = [f"{n} {ph}" for n in names for ph in PN[n] if not np.isfinite(PN[n][ph]['decomp'])]
    print(f"  оценка по разложению не определена (r1 ≥ 0): {len(pn['decomp_undefined'])} фаз из {2 * len(names)} — {', '.join(pn['decomp_undefined'])}")
    AMP = {n: p_amplitude(p, LEAD_II_CHANNEL, S['II'][n]) for n, p in zip(names, paths)}
    a = np.array([AMP[n]['покой'] for n in names]); b = np.array([AMP[n]['наклон'] for n in names]); okk = np.isfinite(a) & np.isfinite(b)
    pn['p_amplitude'] = dict(n=int(okk.sum()), rest=float(np.median(a[okk])), tilt=float(np.median(b[okk])), up=int((b[okk] > a[okk]).sum()), p=float(stats.wilcoxon(a[okk], b[okk])[1]))
    v = pn['p_amplitude']; print(f"  размах среднего зубца P (отведение II): покой {v['rest']:.0f} → наклон {v['tilt']:.0f}, выше при наклоне у {v['up']}/{v['n']}, p = {v['p']:.4f}")
    summary['noise_by_phase'] = pn
    print(f"Погрешность PR (медианы): разложение — {summary['noise']['decomp']['noise_sd']:.2f} мс (без уточнения {summary['noise']['decomp_raw']['noise_sd']:.2f}); "
          f"разница отведений II и III — {summary['noise']['crosslead']:.2f} мс (без уточнения {summary['noise']['crosslead_raw']:.2f}); "
          f"собственная изменчивость разностей — {summary['noise']['decomp']['own_sd']:.2f} мс; "
          f"по разложению не определена (r1 ≥ 0): {', '.join(summary['noise']['decomp_undefined']) or 'нет'} (без уточнения: {', '.join(summary['noise']['decomp_raw_undefined']) or 'нет'})")
    lf, rows = lfhf_analysis(U); summary['lfhf'] = lf
    for k in ('RR', 'PP', 'PR'):
        v = lf[k]; print(f"НЧ/ВЧ {k}: {v['rest']:.2f} → {v['tilt']:.2f} (×{v['ratio']:.2f}), рост у {v['up']}/{v['n']}, p = {v['p']:.4f}")
    print(f"Прямое сравнение сдвигов: RR–PR p = {lf['RR–PR']['p']:.5f} (n = {lf['RR–PR']['n']}), PP–PR p = {lf['PP–PR']['p']:.5f}")
    summary['power'] = power_analysis(U, nz_dec)
    summary['power_crosslead'] = power_analysis(U, nz_cl)     # та же проверка с оценкой шума по разнице отведений II и III
    for lab, P in (('разложение', summary['power']), ('разница II–III', summary['power_crosslead'])):
        v = P['pr_hf_minus_noise']; print(f"ВЧ PR за вычетом шума ({lab}): {v['rest']:.2f} → {v['tilt']:.2f} мс², снижение у {v['down']}/{v['n']}, p = {v['p']:.4f}; "
                                          f"ВЧ PR выше шума в {P['pr_hf_over_noise']['покой']:.1f} (покой) и {P['pr_hf_over_noise']['наклон']:.1f} раза (наклон)")
        for b, bl in (('lf', 'НЧ'), ('hf', 'ВЧ')):
            v = P[f'noise_share_{b}']; print(f"  доля шума в {bl}-мощности PR ({lab}): {v['rest']:.2f} → {v['tilt']:.2f}, меньше при наклоне у {v['down']}/{v['n']}, p = {v['p']:.4f}")
        v = P['pr_lfhf_minus_noise']; w_ = P['rr_vs_pr_minus_noise']
        print(f"  НЧ/ВЧ PR за вычетом шума ({lab}): {v['rest']:.2f} → {v['tilt']:.2f} (×{v['ratio']:.2f}), рост у {v['up']}/{v['n']}, p = {v['p']:.4f}; "
              f"сдвиг RR против сдвига PR: p = {w_['p']:.4f}, у RR сильнее у {w_['rr_stronger']}/{w_['n']}")
    ch_main, rec = coherence_analysis(U); summary['coherence'] = ch_main
    for bn in ('LF', 'HF'):
        v = ch_main['изменение'][bn]; print(f"Когерентность {bn}: {v['rest']:.2f} → {v['tilt']:.2f}, рост у {v['up']}/{v['n']}, p = {v['p']:.4f}")
    summary['coherence_corrected'] = {}
    for lab, P in (('разложение', summary['power']), ('разница II–III', summary['power_crosslead'])):
        cc = coherence_noise_corrected(rec, P['shares']); summary['coherence_corrected'][lab] = cc
        for bn in ('LF', 'HF'):
            v = cc[bn]; print(f"  когерентность {bn} с поправкой на шум ({lab}): {v['rest']:.2f} → {v['tilt']:.2f}, рост у {v['up']}/{v['n']}, p = {v['p']:.4f}")
    rob = {}
    for lab, UU, seg in (('окно 16 с', U, 64), ('окно 64 с', U, 256), ('лучшее отведение записи', prepare(S['best']), COH_SEG),
                         ('без уточнения положений R и P', prepare(S['II_raw']), COH_SEG)):
        rob[lab] = coherence_analysis(UU, seg=seg, with_null=False)[0]['изменение']
    summary['coherence_robustness'] = rob
    summary['peaks'] = peak_dynamics(U, nz_cl); summary['peaks_decomp'] = peak_dynamics(U, nz_dec)
    for lab, PK in (('разница II–III', summary['peaks']), ('разложение', summary['peaks_decomp'])):
        for ph, v in PK.items():
            print(f"Пик ВЧ, {ph} (погрешность: {lab}): СКО RR {v['sd_rr']:.3f}, PR {v['sd_pr']:.3f}, искусственный PR {v['sd_sim']:.3f} Гц; "
                  f"PR сильнее искусственного у {v['pr_gt_sim']}/{v['n']} (знаковый критерий p = {v['p_sign']:.3f}); "
                  f"связь траекторий {v['rho']:+.2f}, положительна у {v['rho_pos']}/{v['n_rho']}")
    summary['ljung_box_white_noise'] = ljung_box_check()
    summary['ljung_box_pr_diffs'] = int(sum(ljung_box_p(np.diff(S['II'][n]['pr'][np.isfinite(S['II'][n]['pr'])])) < 0.05 for n in names))
    var = {'основной расчёт': lf,
           'окна: фазы целиком, с переходным процессом': lfhf_analysis(prepare(S['II'], windows='whole'))[0],
           'детектор P только с основным порогом': lfhf_analysis(prepare(S['II_main_thr']))[0],
           'без требования охвата ряда': lfhf_analysis(prepare(S['II'], cov=0.0))[0],
           'без исправления артефактов и выбросов': lfhf_analysis(prepare(S['II'], clean=False))[0],
           'без уточнения положений R и P': lfhf_analysis(prepare(S['II_raw']))[0],
           'лучшее отведение записи': lfhf_analysis(prepare(S['best']))[0]}
    summary['variants'] = var
    for lab, v in var.items():
        print(f"  {lab:46s} RR p={v['RR']['p']:.4f} | PP p={v['PP']['p']:.4f} (n={v['PP']['n']}) | PR p={v['PR']['p']:.4f} (n={v['PR']['n']}) | RR–PR p={v['RR–PR']['p']:.4f}")
    fig_lfhf(rows, os.path.join(out_dir, 'p4_fig1_lfhf.png')); fig_coh(rec, os.path.join(out_dir, 'p4_fig2_coherence.png'))
    with open(os.path.join(out_dir, 'p4_per_record.csv'), 'w', newline='', encoding='utf-8') as fh:
        w = csv.writer(fh); w.writerow(['запись', 'фаза', 'НЧ/ВЧ RR', 'НЧ/ВЧ PP', 'НЧ/ВЧ PR', 'когерентность НЧ', 'когерентность ВЧ', 'погрешность PR (разложение), мс', 'погрешность PR (II–III), мс', 'погрешность PR в фазе (разложение), мс', 'погрешность PR в фазе (II–III), мс'])
        for n in names:
            for ph in ('покой', 'наклон'):
                c = rec.get((n, ph), {}); r = rows[n][ph]
                w.writerow([n, ph] + [('' if r[k] is None else round(r[k], 4)) for k in ('RR', 'PP', 'PR')] +
                           [('' if not c else round(c[b], 4)) for b in ('LF', 'HF')] + [round(dec[n]['noise_sd'], 3), round(cl[n], 3), round(PN[n][ph]['decomp'], 3), round(PN[n][ph]['crosslead'], 3)])
    json.dump(_clean(summary), open(os.path.join(out_dir, 'p4_summary.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
    print(f'Результаты сохранены в папку {out_dir}')
    return summary


def _in_notebook():
    try:
        get_ipython  # noqa: F821
        return True
    except NameError:
        return False


if __name__ == '__main__' and not _in_notebook():
    ap = argparse.ArgumentParser(description='П.4: динамика частотных составляющих RR и PR')
    ap.add_argument('data_dir'); ap.add_argument('--out', default='results_p4')
    a = ap.parse_args(); run_p4(a.data_dir, a.out)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 20.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyedflib: filename=pyedflib-0.1.42-cp313-cp313-linux_x86_64.whl size=2852050 sha256=d0e0b4b100839aa1ed3f97a15942f48db2ccbc4799d2a61ffc08469200734ff3
  Stored in directory: /root/.cache/pip/wheels/ac/16/3e/5299c8a6b023bfd3d7204240ca5de940a49bede1edb30015ab
Successfully built pyedflib


In [2]:
run_p4('/content/data/')

Записей: 20. Строю ряды (отведения II и III, варианты измерения)...
Погрешность PR по фазам (разложение, записи с обеими фазами): покой 1.54 → наклон 1.12 мс, меньше при наклоне у 12/14, p = 0.0052
Погрешность PR по фазам (разница II–III, записи с обеими фазами): покой 1.96 → наклон 0.81 мс, меньше при наклоне у 15/19, p = 0.0062
  оценка по разложению не определена (r1 ≥ 0): 7 фаз из 40 — 0011 наклон, 0020 наклон, 0023 наклон, 0026 покой, 0026 наклон, 0029 покой, 0029 наклон
  размах среднего зубца P (отведение II): покой 156 → наклон 182, выше при наклоне у 15/20, p = 0.0017
Погрешность PR (медианы): разложение — 1.44 мс (без уточнения 2.05); разница отведений II и III — 1.45 мс (без уточнения 2.76); собственная изменчивость разностей — 1.58 мс; по разложению не определена (r1 ≥ 0): 0023, 0029 (без уточнения: 0029)
НЧ/ВЧ RR: 0.55 → 1.49 (×3.58), рост у 19/20, p = 0.0000
НЧ/ВЧ PP: 0.69 → 1.65 (×3.70), рост у 18/19, p = 0.0002
НЧ/ВЧ PR: 0.70 → 0.89 (×0.91), рост у 8/19, p = 0.8906
Прям

{'noise': {'decomp_undefined': ['0023', '0029'],
  'decomp_raw_undefined': ['0029'],
  'decomp': {'r1': -0.2603388633439505,
   'noise_sd': 1.4434943368114566,
   'own_sd': 1.5792759845219184},
  'decomp_raw': {'r1': -0.3471837852977341,
   'noise_sd': 2.0499098474838116,
   'own_sd': 1.6918260975821704},
  'crosslead': 1.4537203567593273,
  'crosslead_raw': 2.7620511858702113},
 'noise_by_phase': {'records': ['0011',
   '0012',
   '0013',
   '0014',
   '0015',
   '0016',
   '0017',
   '0019',
   '0020',
   '0021',
   '0022',
   '0023',
   '0024',
   '0025',
   '0026',
   '0027',
   '0028',
   '0029',
   '0030'],
  'decomp': {'n': 14,
   'rest': 1.5387855747481807,
   'tilt': 1.1203130920038151,
   'down': 12,
   'p': 0.0052490234375},
  'crosslead': {'n': 19,
   'rest': 1.9614660886011082,
   'tilt': 0.8060076232420388,
   'down': 15,
   'p': 0.0061798095703125},
  'decomp_undefined': ['0011 наклон',
   '0020 наклон',
   '0023 наклон',
   '0026 покой',
   '0026 наклон',
   '0029 покой